# Waze × KMZ — Detector de Obras en Vía

**Flujo:**
1. Instalar dependencias
2. Configuración central
3. Leer y filtrar el JSON de Waze
4. Parsear el archivo KMZ
5. Análisis de proximidad
6. Generar mapa Folium interactivo
7. `main()` — orquesta todo

## 1. Dependencias

In [ ]:
# Ejecutar solo la primera vez (o en entornos nuevos)
%pip install requests folium shapely lxml numpy -q

In [ ]:
import json
import zipfile
import math
from datetime import datetime
from pathlib import Path

import folium
from lxml import etree
from shapely.geometry import Point, LineString
from shapely.ops import nearest_points, unary_union

print('✓ Imports OK')

## 2. Configuración central

In [ ]:
CONFIG = {
    # ── Geografía ──────────────────────────────
    "center_lat":        10.39972,
    "center_lon":       -75.51444,
    "search_radius_km": 10,

    # ── Archivos ───────────────────────────────
    # JSON acumulado generado por collector.py
    "json_path":  "data/waze_partnerhub.json",
    # KMZ con las líneas de tubería/vía
    "kmz_path":   "KMZ/TuberiaAcero_Cartagena.kmz",
    # HTML de salida
    "output_html": "mapa_waze_hazards.html",

    # ── Análisis ───────────────────────────────
    # Radio de área de influencia en metros
    "buffer_radius_m": 200,
}

print('✓ Configuración cargada')
print(f"  JSON  : {CONFIG['json_path']}")
print(f"  KMZ   : {CONFIG['kmz_path']}")
print(f"  Salida: {CONFIG['output_html']}")

## 3. Leer y filtrar el JSON de Waze

In [ ]:
# ── Tipos y subtipos de interés ─────────────────────────────────────
TARGET_TYPES = {"HAZARD", "ROAD_CLOSED"}

TARGET_SUBTYPES = {
    "",                              # sin subtipo (campo vacío)
    None,                            # campo ausente
    "HAZARD_ON_ROAD_CONSTRUCTION",
    "HAZARD_ON_ROAD_POT_HOLE",
    "ROAD_CLOSED_CONSTRUCTION",
}

# Ajusta TARGET_SUBTYPES según los eventos que te interesen filtrar

In [ ]:
def load_waze_store(json_path: str) -> dict:
    """
    Lee el archivo JSON acumulativo generado por collector.py.
    Estructura esperada:
      { "meta": {...}, "records": { "<uuid>": { ...alert... } } }
    """
    path = Path(json_path)
    if not path.exists():
        print(f'[JSON] ✗ Archivo no encontrado: {json_path}')
        return {}

    with open(path, 'r', encoding='utf-8') as f:
        store = json.load(f)

    meta    = store.get('meta', {})
    records = store.get('records', {})
    print(f'[JSON] ✓ Cargado: {json_path}')
    print(f'       Última actualización : {meta.get("last_updated", "—")}')
    print(f'       Total registros       : {meta.get("total_records", len(records))}')
    return store


def filter_hazards(store: dict) -> list:
    """
    Extrae los registros del store y filtra por TARGET_TYPES / TARGET_SUBTYPES.
    Retorna lista de alertas (dicts) lista para análisis.
    """
    records = store.get('records', {})
    all_alerts = list(records.values())

    filtered = [
        a for a in all_alerts
        if  a.get('type')    in TARGET_TYPES
        and a.get('subtype') in TARGET_SUBTYPES
    ]

    print(f'\n[FILTRO] {len(all_alerts)} registros totales '
          f'→ {len(filtered)} eventos seleccionados')
    return filtered

## 4. Parsear el KMZ

In [ ]:
def parse_coordinates_text(text: str) -> list:
    """Parsea texto de <coordinates> KML → lista de (lon, lat)."""
    coords = []
    for token in text.strip().split():
        parts = token.split(',')
        if len(parts) >= 2:
            try:
                lon, lat = float(parts[0]), float(parts[1])
                coords.append((lon, lat))
            except ValueError:
                pass
    return coords


def parse_kmz(kmz_path: str) -> tuple:
    """
    Lee un KMZ y extrae todos los LineString como objetos Shapely.
    Retorna: (lista de LineString, lista de nombres)
    """
    print(f'\n[KMZ] Leyendo: {kmz_path}')
    lines, names = [], []

    try:
        with zipfile.ZipFile(kmz_path, 'r') as kmz:
            kml_files = [f for f in kmz.namelist() if f.lower().endswith('.kml')]
            if not kml_files:
                print('[KMZ] ✗ No se encontró archivo .kml dentro del KMZ')
                return [], []

            for kml_file in kml_files:
                print(f'[KMZ] Procesando: {kml_file}')
                with kmz.open(kml_file) as kf:
                    tree = etree.parse(kf)
                    root = tree.getroot()

                # Detectar namespace automáticamente
                tag = root.tag
                ns  = tag[1:tag.index('}')] if '}' in tag else ''
                prefix = '{' + ns + '}' if ns else ''

                placemark_tag   = f'{prefix}Placemark'
                name_tag        = f'{prefix}name'
                linestring_tag  = f'{prefix}LineString'
                coordinates_tag = f'{prefix}coordinates'

                for pm in root.iter(placemark_tag):
                    name_el = pm.find(name_tag)
                    pm_name = (
                        name_el.text.strip()
                        if name_el is not None and name_el.text
                        else 'Sin nombre'
                    )
                    for ls in pm.iter(linestring_tag):
                        coord_el = ls.find(coordinates_tag)
                        if coord_el is not None and coord_el.text:
                            coords = parse_coordinates_text(coord_el.text)
                            if len(coords) >= 2:
                                lines.append(LineString(coords))
                                names.append(pm_name)

        print(f'[KMZ] ✓ {len(lines)} líneas extraídas')
        return lines, names

    except FileNotFoundError:
        print(f'[KMZ] ✗ Archivo no encontrado: {kmz_path}')
        return [], []
    except Exception as e:
        print(f'[KMZ] ✗ Error: {e}')
        return [], []

## 5. Análisis de proximidad

In [ ]:
# Factor de conversión aproximado grados → metros en Colombia
DEG_TO_M = 111_000   # ~111 km por grado


def degrees_to_meters(deg: float) -> float:
    return deg * DEG_TO_M


def analyze_proximity(hazards: list, kmz_lines: list, kmz_names: list) -> list:
    """
    Enriquece cada hazard con:
      _dist_m     : distancia en metros al LineString más cercano
      _line_name  : nombre de esa línea
      _nearest_pt : punto exacto en la línea (Shapely Point)
    Retorna la lista ordenada de menor a mayor distancia.
    """
    if not kmz_lines:
        print('[PROXIMIDAD] ⚠️ Sin líneas KMZ — distancias no calculadas')
        for h in hazards:
            h['_dist_m']    = None
            h['_line_name'] = 'N/A'
            h['_nearest_pt']= None
        return hazards

    kmz_union = unary_union(kmz_lines)

    for h in hazards:
        lon = h.get('location', {}).get('x')
        lat = h.get('location', {}).get('y')
        if lon is None or lat is None:
            h['_dist_m'] = None
            h['_line_name'] = 'N/A'
            h['_nearest_pt'] = None
            continue

        pt = Point(lon, lat)

        # Distancia al conjunto unificado
        h['_dist_m'] = round(degrees_to_meters(kmz_union.distance(pt)))

        # Línea individual más cercana
        min_dist, min_name, min_point = float('inf'), 'N/A', None
        for line, name in zip(kmz_lines, kmz_names):
            d = line.distance(pt)
            if d < min_dist:
                min_dist  = d
                min_name  = name
                _, nearest = nearest_points(pt, line)
                min_point  = nearest

        h['_line_name']  = min_name
        h['_nearest_pt'] = min_point   # Shapely Point (lon, lat)

    # Ordenar por distancia ascendente
    hazards.sort(key=lambda h: h['_dist_m'] if h['_dist_m'] is not None else 9e9)

    print(f'[PROXIMIDAD] ✓ {len(hazards)} hazards analizados')
    return hazards


def print_ranking(hazards: list, top_n: int = 10):
    """Imprime el ranking de hazards más cercanos a las líneas KMZ."""
    print('\n' + '─' * 65)
    print(f'  TOP {top_n} EVENTOS más cercanos a líneas KMZ ({len(hazards)} total)')
    print('─' * 65)
    for i, h in enumerate(hazards[:top_n], 1):
        dist   = h.get('_dist_m')
        calle  = h.get('street', '—') or '—'
        lname  = h.get('_line_name', 'N/A')
        tipo   = h.get('type', '—')
        dist_s = f'{dist:>6,} m' if dist is not None else '    N/A'
        print(f'  #{i:02d}  {calle[:32]:<32}  [{tipo:<12}]  {dist_s}  →  {lname}')
    if len(hazards) > top_n:
        print(f'       ... y {len(hazards) - top_n} más')
    print('─' * 65)

## 6. Mapa Folium

In [ ]:
def ts_to_str(millis):
    """Convierte timestamp en milisegundos a string legible."""
    try:
        return datetime.fromtimestamp(millis / 1000).strftime('%Y-%m-%d %H:%M')
    except Exception:
        return '—'

In [ ]:
def create_folium_map(
    center_lat: float,
    center_lon: float,
    hazards: list,
    kmz_lines: list,
    kmz_names: list,
    buffer_m: int = 300,
    search_radius_km: int = 10,
):
    """Crea y retorna el mapa Folium interactivo."""

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=13,
        tiles='CartoDB positron',
    )

    folium.TileLayer('OpenStreetMap',       name='OpenStreetMap').add_to(m)
    folium.TileLayer('CartoDB dark_matter', name='Dark (noche)').add_to(m)

    # ── Grupos de capas ─────────────────────────────────
    grp_kmz     = folium.FeatureGroup(name='🛣️ Líneas KMZ',             show=True)
    grp_hazards = folium.FeatureGroup(name='⚠️ Eventos Waze',           show=True)
    grp_buffer  = folium.FeatureGroup(name=f'🔵 Área influencia ({buffer_m}m)', show=True)
    grp_links   = folium.FeatureGroup(name='🔗 Enlace al punto cercano', show=True)

    # ── 1. Líneas KMZ ────────────────────────────────────
    colors_kmz = ['#0066ff', '#ff6600', '#009933', '#9900cc', '#cc0000']
    for i, (line, name) in enumerate(zip(kmz_lines, kmz_names)):
        coords_ll = [(lat, lon) for lon, lat in line.coords]
        color = colors_kmz[i % len(colors_kmz)]
        folium.PolyLine(
            coords_ll, color=color, weight=5, opacity=0.85,
            tooltip=f'<b>{name}</b>',
            popup=folium.Popup(
                f'<b>Vía:</b> {name}<br>'
                f'<b>Segmentos:</b> {len(line.coords) - 1}<br>'
                f'<b>Longitud aprox:</b> {degrees_to_meters(line.length):.0f} m',
                max_width=250,
            ),
        ).add_to(grp_kmz)

    # ── 2. Hazards ───────────────────────────────────────
    for i, h in enumerate(hazards):
        lat       = h.get('location', {}).get('y')
        lon       = h.get('location', {}).get('x')
        dist_m    = h.get('_dist_m')
        line_name = h.get('_line_name', 'N/A')
        nearest   = h.get('_nearest_pt')
        street    = h.get('street',      '—') or '—'
        city      = h.get('city',        '—') or '—'
        tipo      = h.get('type',        '—')
        subtipo   = h.get('subtype',     '—') or '(sin subtipo)'
        reliability = h.get('reliability',  '—')
        thumbs_up   = h.get('reportRating', 0)
        pub_millis  = h.get('pubMillis',    0)
        collected   = h.get('_collected_at', '—')
        rank        = i + 1

        # Color según distancia
        if dist_m is None:    icon_color = 'gray'
        elif dist_m <= 100:   icon_color = 'red'
        elif dist_m <= 500:   icon_color = 'orange'
        else:                 icon_color = 'blue'

        dist_label = f'{dist_m:,} m' if dist_m is not None else 'N/A'

        popup_html = f"""
        <div style="font-family:Arial;font-size:13px;min-width:250px">
          <h4 style="margin:0;color:#d35400">⚠️ Evento Waze #{rank}</h4>
          <hr style="margin:6px 0">
          <b>Tipo:</b> {tipo}<br>
          <b>Subtipo:</b> {subtipo}<br>
          <b>Calle:</b> {street}<br>
          <b>Ciudad:</b> {city}<br>
          <b>Confiabilidad:</b> {reliability}/10 &nbsp;👍 {thumbs_up}<br>
          <b>Publicado:</b> {ts_to_str(pub_millis)}<br>
          <b>Capturado:</b> {collected[:16]}<br>
          <hr style="margin:6px 0">
          <b>📏 Distancia a vía KMZ:</b>
            <span style="color:#2980b9"><b>{dist_label}</b></span><br>
          <b>🛣️ Vía más cercana:</b> {line_name}<br>
          <hr style="margin:6px 0">
          <small style="color:#888">UUID: {h.get('uuid','—')}</small>
        </div>
        """

        # Área de influencia
        folium.Circle(
            location=[lat, lon], radius=buffer_m,
            color=icon_color, fill=True,
            fill_color=icon_color, fill_opacity=0.12, weight=1.5,
        ).add_to(grp_buffer)

        # Marcador
        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color=icon_color, icon='warning-sign', prefix='glyphicon'),
            tooltip=f'⚠️ {street} | {dist_label} de \'{line_name}\'',
            popup=folium.Popup(popup_html, max_width=300),
        ).add_to(grp_hazards)

        # Línea punteada al punto más cercano del KMZ
        if nearest is not None:
            folium.PolyLine(
                locations=[[lat, lon], [nearest.y, nearest.x]],
                color='#e74c3c', weight=1.5, dash_array='6 4',
                opacity=0.7, tooltip=f'Dist: {dist_label}',
            ).add_to(grp_links)
            folium.CircleMarker(
                location=[nearest.y, nearest.x],
                radius=4, color='#e74c3c', fill=True,
                fill_color='white', fill_opacity=1, weight=2,
            ).add_to(grp_links)

    # ── Agregar grupos al mapa ───────────────────────────
    grp_kmz.add_to(m)
    grp_buffer.add_to(m)
    grp_links.add_to(m)
    grp_hazards.add_to(m)

    # ── Marcador de centro de búsqueda ───────────────────
    folium.Marker(
        location=[center_lat, center_lon],
        icon=folium.Icon(color='green', icon='home', prefix='glyphicon'),
        tooltip='Centro de búsqueda',
    ).add_to(m)

    folium.Circle(
        location=[center_lat, center_lon],
        radius=search_radius_km * 1000,
        color='#27ae60', fill=False, weight=2,
        dash_array='8 4', opacity=0.5,
        tooltip=f'Área de búsqueda ({search_radius_km} km)',
    ).add_to(m)

    # ── Panel de estadísticas ────────────────────────────
    n_kmz    = len(kmz_lines)
    n_haz    = len(hazards)
    near_100 = sum(1 for h in hazards if (h.get('_dist_m') or 9999) <= 100)
    near_500 = sum(1 for h in hazards if (h.get('_dist_m') or 9999) <= 500)
    valid_d  = [h['_dist_m'] for h in hazards if h.get('_dist_m') is not None]
    avg_dist = sum(valid_d) / len(valid_d) if valid_d else 0
    ts       = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    legend_html = f"""
    <div style="
        position:fixed; bottom:30px; left:30px; z-index:1000;
        background:white; border-radius:10px; padding:16px;
        box-shadow:0 4px 20px rgba(0,0,0,0.2);
        font-family:Arial; font-size:13px; min-width:260px;
        border-left:4px solid #e74c3c;
    ">
      <h4 style="margin:0 0 10px;color:#2c3e50">📊 Resumen</h4>
      <b>Actualizado:</b> {ts}<br>
      <hr style="margin:8px 0">
      <b>🛣️ Líneas KMZ:</b> {n_kmz}<br>
      <b>⚠️ Eventos detectados:</b> {n_haz}<br>
      <b>🔴 ≤ 100 m de vía:</b> {near_100}<br>
      <b>🟠 ≤ 500 m de vía:</b> {near_500}<br>
      <b>📏 Distancia promedio:</b> {avg_dist:,.0f} m<br>
      <b>🔵 Buffer activo:</b> {buffer_m} m<br>
      <hr style="margin:8px 0">
      <small style="color:#888">
        Radio búsqueda: {search_radius_km} km<br>
        Centro: {center_lat:.4f}, {center_lon:.4f}
      </small>
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

    # ── Leyenda de colores ───────────────────────────────
    color_legend = """
    <div style="
        position:fixed; bottom:30px; right:30px; z-index:1000;
        background:white; border-radius:10px; padding:14px;
        box-shadow:0 4px 20px rgba(0,0,0,0.2);
        font-family:Arial; font-size:12px;
    ">
      <b>Distancia a vía KMZ</b><br>
      <span style="color:#e74c3c">●</span> ≤ 100 m (crítico)<br>
      <span style="color:#e67e22">●</span> ≤ 500 m (cercano)<br>
      <span style="color:#2980b9">●</span> &gt; 500 m (lejano)<br>
      <hr style="margin:6px 0">
      <span style="color:#0066ff">━━</span> Línea KMZ<br>
      <span style="color:#e74c3c">- -</span> Enlace punto cercano
    </div>
    """
    m.get_root().html.add_child(folium.Element(color_legend))

    folium.LayerControl(collapsed=False).add_to(m)
    return m

## 7. Main — Orquestador

In [ ]:
def main():
    print('=' * 65)
    print('   WAZE × KMZ — Detector de Obras en Vía | Cartagena')
    print('=' * 65)

    # ── Paso 1: Leer JSON acumulado ──────────────────────
    print('\n── PASO 1: Cargar JSON de Waze ─────────────────────────')
    store = load_waze_store(CONFIG['json_path'])

    # ── Paso 2: Filtrar eventos de interés ───────────────
    print('\n── PASO 2: Filtrar eventos ──────────────────────────────')
    hazards = filter_hazards(store)

    if not hazards:
        print('\n[WARN] No se encontraron eventos con los filtros actuales.')
        print('       Revisa TARGET_TYPES y TARGET_SUBTYPES en la celda de configuración.')
        return

    # ── Paso 3: Parsear KMZ ──────────────────────────────
    print('\n── PASO 3: Parsear KMZ ──────────────────────────────────')
    kmz_lines, kmz_names = parse_kmz(CONFIG['kmz_path'])

    # ── Paso 4: Análisis de proximidad ───────────────────
    print('\n── PASO 4: Análisis de proximidad ───────────────────────')
    hazards = analyze_proximity(hazards, kmz_lines, kmz_names)
    print_ranking(hazards, top_n=10)

    # ── Paso 5: Generar mapa ─────────────────────────────
    print('\n── PASO 5: Generar mapa ─────────────────────────────────')
    mapa = create_folium_map(
        center_lat       = CONFIG['center_lat'],
        center_lon       = CONFIG['center_lon'],
        hazards          = hazards,
        kmz_lines        = kmz_lines,
        kmz_names        = kmz_names,
        buffer_m         = CONFIG['buffer_radius_m'],
        search_radius_km = CONFIG['search_radius_km'],
    )

    output = CONFIG['output_html']
    mapa.save(output)
    print(f'[MAPA] ✓ Guardado: {output}')

    print('\n' + '=' * 65)
    print(f'  ➜  Abre en tu navegador: {output}')
    print('=' * 65)

    return mapa


# ── Ejecutar ─────────────────────────────────────────────
mapa = main()

## 8. Visualizar el mapa inline (opcional en Jupyter)

In [ ]:
# Muestra el mapa directamente en el notebook
# (solo funciona dentro de Jupyter / VS Code / Colab)
# mapa